# Experiment with Calibration

This notebook demonstrates isotonic calibration and its effect on model predictions.

In [ ]:
import sys
from pathlib import Path

phase_dir = Path().cwd().parent
sys.path.insert(0, str(phase_dir / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss

from phase01_lightgbm_baseline.data_loader import DataLoader
from phase01_lightgbm_baseline.lightgbm_baseline import LightGBMWrapper
from src.utils.calibration import IsotonicCalibrator

## Load Data and Train Model

In [ ]:
loader = DataLoader()
train_df, test_df = loader.load_data(sample_limit=500)

baseline_features = ["TransactionAmt", "time_since_creation", "n_prior_transactions"]
X_train = train_df[baseline_features].values
y_train = train_df["isFraud"].values
X_test = test_df[baseline_features].values
y_test = test_df["isFraud"].values

model = LightGBMWrapper(random_state=42)
model.train(X_train, y_train, X_test, y_train)

raw_probs = model.predict(X_test)
print(f"Raw predictions - Mean: {raw_probs.mean():.4f}, Std: {raw_probs.std():.4f}")

## Apply Calibration

In [ ]:
calibrator = IsotonicCalibrator()
calibrator.fit(raw_probs, y_test)
calibrated_probs = calibrator.predict(raw_probs)

print(f"Calibrated predictions - Mean: {calibrated_probs.mean():.4f}, Std: {calibrated_probs.std():.4f}")

brier_raw = brier_score_loss(y_test, raw_probs)
brier_cal = brier_score_loss(y_test, calibrated_probs)

print(f"\nBrier score (raw): {brier_raw:.4f}")
print(f"Brier score (calibrated): {brier_cal:.4f}")
print(f"Improvement: {brier_raw - brier_cal:+.4f}")

## Visualize Calibration Curves

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

prob_true_raw, prob_pred_raw = calibration_curve(y_test, raw_probs, n_bins=10)
ax.plot(prob_pred_raw, prob_true_raw, marker='o', label='Raw', linewidth=2)

prob_true_cal, prob_pred_cal = calibration_curve(y_test, calibrated_probs, n_bins=10)
ax.plot(prob_pred_cal, prob_true_cal, marker='s', label='Calibrated', linewidth=2)

ax.plot([0, 1], [0, 1], '--', color='gray', label='Perfect calibration')

ax.set_xlabel('Mean predicted probability')
ax.set_ylabel('Fraction of positives')
ax.set_title('Calibration Curve')
ax.legend()
plt.tight_layout()
plt.show()

## Compare Distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(raw_probs, bins=30, alpha=0.7, label='Raw', edgecolor='black')
axes[0].hist(calibrated_probs, bins=30, alpha=0.7, label='Calibrated', edgecolor='black')
axes[0].set_xlabel('Predicted Probability')
axes[0].set_ylabel('Count')
axes[0].set_title('Prediction Distribution')
axes[0].legend()

axes[1].scatter(raw_probs, calibrated_probs, alpha=0.5, s=10)
axes[1].plot([0, 1], [0, 1], '--', color='gray')
axes[1].set_xlabel('Raw Probability')
axes[1].set_ylabel('Calibrated Probability')
axes[1].set_title('Raw vs Calibrated')

plt.tight_layout()
plt.show()

## Summary

Calibration adjusts the predicted probabilities to better match the true fraud rates. Lower Brier scores indicate better calibration. The calibration curve shows how close the model is to perfect calibration.